#### Intall & Setup UV

In [ ]:
%%bash 
curl -LsSf https://astral.sh/uv/install.sh | sh

In [ ]:
#%pip install -U crewai crewai_tools 'crewai[tools, litellm]' unstructured

##### Setup and Run ollama locally

```bash

#
curl -fsSL https://ollama.com/install.sh | sh
#
sudo systemctl status ollama
sudo systemctl start ollama
sudo systemctl stop ollama

#
docker run -d -v ollama:/root/.ollama -p 11434:11434 --name ollama ollama/ollama

#
ollama list

# 4K
/set parameter num_ctx 4096
# 8K
/set parameter num_ctx 8192
# 16K
/set parameter num_ctx 16384  
# 32K
/set parameter num_ctx 32768

#
# models : sudo ls -l /usr/share/ollama/.ollama/models
#
ollama pull gemma4:latest
ollama run gemma4:latest 
ollama stop gemma4:latest

# Pull and run the latest models in one command
ollama run qwen3.5
# For the fastest local inference:
ollama run gemma4:26b
# For the latest reasoning models:
ollama run deepseek-v3.2-exp:7b
# For agentic AI with Kimi K2.6:
ollama launch kimi --model kimi-k2.6:cloud

curl -X POST http://localhost:11434/api/generate -d '{"model": "gemma4:latest", "prompt":"Why is the sky blue?","options":{"num_ctx":4096} }'


#
# embeddings
#
ollama pull nomic-embed-text:latest
ollama run nomic-embed-text:latest
curl http://localhost:11434/v1/embeddings -d '{"model": "nomic-embed-text:latest", "prompt": "Why is the sky blue?"}'
curl http://localhost:11434/api/embeddings -d '{"model": "nomic-embed-text:latest", "prompt": "Why is the sky blue?"}'


# http://localhost:11434
# http://localhost:11434/v1/models

curl http://localhost:11434/api/ps

curl http://localhost:11434/api/embed -d '{
  "model": "nomic-embed-text:latest",
  "input": "Why is the sky blue?"
}'

curl http://localhost:11434/api/show -d '{ "model": "gemma4:latest" }'


ollama rm gemma4:latest
# /usr/share/ollama/.ollama/models
curl -X DELETE http://localhost:11434/api/delete -d '{ "model": "gemma4:latest" }'
```

```
FROM llama3

# Set parameters

PARAMETER temperature 0.8
PARAMETER stop Result

# Sets a custom system message to specify the behavior of the chat assistant

# Leaving it blank for now.

SYSTEM """"""
```
```bash
ollama create crewai-llama3 -f .\Modelfile
```

#### Seup Docker Model Runner Locally

```bash
sudo -S apt-get update
sudo -S apt-get install docker-model-plugin

docker model reinstall-runner --backend vllm --gpu cuda

docker model pull ai/gpt-oss:20B
docker model run --detach ai/gpt-oss:20B
docker model stop ai/gpt-oss:20B

docker model pull ai/smollm2
docker model run --detach ai/smollm2
docker model stop ai/smollm2

# Pull a small, efficient model (good for getting started)
docker model pull ai/llama3.2:1B-Q8_0
docker model run --detach ai/llama3.2:1B-Q8_0

# Pull a larger, more capable model
docker model pull ai/llama3.2:3B-Q4_K_M

# Pull a code-specialized model
docker model pull ai/codellama:7B-Q4_K_M

docker model pull ai/llama3.2
docker model pull ai/llama3.2:1B-Q8_0
docker model run --detach ai/llama3.2
docker model stop ai/llama3.2

# List all downloaded models with sizes
docker model list

# Show detailed information about a model
docker model inspect ai/llama3.2:1B-Q8_0

# Remove a model to free disk space
docker model rm ai/llama3.2:1B-Q8_0

# Remove all unused models
docker model prune

docker model uninstall-runner --models
docker model uninstall-runner --images

http://localhost:12434/engines/v1/models

http://vmware-ubuntu.sandbox.net:12434
# The model is now serving on a local endpoint
# Default: http://localhost:12434/v1

#
curl http://localhost:12434/engines/v1/embeddings \
  -H "Content-Type: application/json" \
  -d '{
    "model": "ai/qwen3-embedding",
    "input": "A dog is an animal"
  }'
  
curl http://localhost:12434/api/chat \
  -H "Content-Type: application/json" \
  -d '{"model": "ai/smollm2", "messages": [{"role": "user", "content": "Hello!"}]}'

# Send a chat completion request using curl
curl http://localhost:12434/engines/v1/chat/completions \
  -H "Content-Type: application/json" \
  -d '{
    "model": "ai/llama3.2:1B-Q8_0",
    "messages": [
      {"role": "system", "content": "You are a helpful assistant."},
      {"role": "user", "content": "Explain Docker volumes in 3 sentences."}
    ],
    "temperature": 0.7,
    "max_tokens": 256
  }'
```

```yaml
# ./docker-compose.yaml
services:
  app:
    image: my-app
    models:
      llm:
        endpoint_var: AI_MODEL_URL
        model_var: AI_MODEL_NAME
      embedding-model:
        endpoint_var: EMBEDDING_URL
        model_var: EMBEDDING_NAME

models:
  dev_model:
    model: ai/model
    context_size: 4096
    runtime_flags:
  llm:
    model: ai/smollm2
    context_size: 4096
    runtime_flags:
      - "--temp 0.1"            # Temperature
      - "--top-p 0.9"           # Top-p sampling
      - "--verbose"             # Set verbosity level to infinity
      - "--verbose-prompt"      # Print a verbose prompt before generation
      - "--log-prefix"          # Enable prefix in log messages
      - "--log-timestamps"      # Enable timestamps in log messages
      - "--log-colors"          # Enable colored logging
  embedding-model:
    model: ai/all-minilm
    context_size: 2048
    runtime_flags:
      - "--embeddings"
```

In [10]:
#Setup config dir

%mkdir -p config

In [ ]:
# %pip install crewai crewai_tools langchain langchain_community langchain_ollama streamlit duckduckgo-search

In [11]:
%%bash

cat <<EOF > config/L001-agents.yaml
researcher:
  role: >
    {topic} Senior Data Researcher
  goal: >
    Uncover cutting-edge developments in {topic}
  backstory: >
    You're a seasoned researcher with a knack for uncovering the latest
    developments in {topic}. You find the most relevant information and
    present it clearly.
EOF


In [12]:
%%bash

cat <<EOF > /home/brijeshdhaker/IdeaProjects/bd-notebooks-module/notebooks/config/L001-tasks.yaml
research_task:
  description: >
    Conduct thorough research about {topic}. Use web search to find current,
    credible information. The current year is 2026.
  expected_output: >
    A markdown report with clear sections: key trends, notable tools or companies,
    and implications. Aim for 800 to 1200 words. No fenced code blocks around the whole document.
  agent: researcher
  output_file: /home/brijeshdhaker/IdeaProjects/bd-notebooks-module/outputs/L001/research_task_report.md
EOF

In [13]:
# src/latest_ai_flow/crews/content_crew/content_crew.py

from typing import List

from crewai import LLM, Agent, Crew, Process, Task
from crewai.agents.agent_builder.base_agent import BaseAgent
from crewai.project import CrewBase, agent, crew, task
from crewai_tools import SerperDevTool

@CrewBase
class ResearchCrew:
  """Single-agent research crew used inside the Flow."""

  agents: List[BaseAgent]
  tasks: List[Task]

  agents_config = "config/L001-agents.yaml"
  tasks_config = "config/L001-tasks.yaml"

  @agent
  def researcher(self) -> Agent:
    return Agent(
      config=self.agents_config["researcher"],  # type: ignore[index]
      verbose=True,
      tools=[SerperDevTool()]
    )

  @task
  def research_task(self) -> Task:
    return Task(
      config=self.tasks_config["research_task"],  # type: ignore[index]
    )

  @crew
  def crew(self) -> Crew:
    return Crew(
      agents=self.agents,
      tasks=self.tasks,
      process=Process.sequential,
      verbose=True,
    )

In [ ]:
# src/latest_ai_flow/main.py
import os
from pydantic import BaseModel
from crewai.flow import Flow, listen, start
#from latest_ai_flow.crews.content_crew.content_crew import ResearchCrew


class ResearchFlowState(BaseModel):
  topic: str = ""
  report: str = ""


class LatestAiFlow(Flow[ResearchFlowState]):
  @start()
  def prepare_topic(self, crewai_trigger_payload: dict | None = None):
    if crewai_trigger_payload:
      self.state.topic = crewai_trigger_payload.get("topic", "AI Agents")
    else:
      self.state.topic = "AI Agents"
    print(f"Topic: {self.state.topic}")

  @listen(prepare_topic)
  def run_research(self):
    result = ResearchCrew().crew().kickoff(inputs={"topic": self.state.topic})
    self.state.report = result.raw
    print("Research crew finished.")

  @listen(run_research)
  def summarize(self):
    print(f"Report path:{os.environ['WORK_DIR']}/outputs/L001/report.md")


def kickoff():
  LatestAiFlow().kickoff()


def plot():
  LatestAiFlow().plot()


#if __name__ == "__main__":
#  kickoff()

In [ ]:
plot()

In [ ]:
kickoff()

In [9]:
import os
from crewai import LLM, Crew, Process, Agent, Task


# Define your agents
researcher = Agent(
    role='Senior Research Analyst',
    goal='Discover new insights',
    backstory="""You're an expert at finding interesting information""",
    verbose=True,
    allow_delegation=False
)

writer = Agent(
    role='Content Writer',
    goal='Write engaging content',
    backstory="""You're a talented writer who simplifies complex information""",
    verbose=True
)

# Create tasks
research_task = Task(
    description='Find interesting facts about AI in healthcare',
    expected_output="A clear, concise summary about AI in healthcare",
    agent=researcher
)

write_task = Task(
    description='Write a short blog post about AI in healthcare',
    expected_output="Write a short blog post about AI in healthcare",
    agent=writer
)

# Form the crew
crew = Crew(
    agents=[researcher, writer],
    tasks=[research_task, write_task],
    process=Process.sequential,
    share_crew=False,
    verbose=True
)

# Execute the crew's tasks
result = crew.kickoff()

print("Here's the result:")
print(result)

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: b7dab55e-93dc-4ec0-b241-5e9b4459c19c                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Find interesting facts about AI in healthcare                                                            │
│  ID: e8467f6e-f6b6-4426-a018-4ff637c04bfb                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Senior Research Analyst                                                                                 │
│                                                                                                                 │
│  Task: Find interesting facts about AI in healthcare                                                            │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Senior Research Analyst                                                                                 │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  ## The Convergence of AI and Healthcare: A Transformative Shift from Reactive to Predictive Medicine           │
│                                                                                                                 │
│  The integration of Artificial Intelligence (AI) into healthcare represents one of the most profound shifts in  │
│  modern medicine, moving the industry from a primarily reactive model (treating existing disease) to a          │
│  proactive, predictive model (forecasting risk and enabling early intervention). As a research analyst, the     │
│  key insight is that AI is not merely a tool for efficiency, but a foundational layer that radically            │
│  accelerates scientific discovery and radically improves diagnostic precision.                                  │
│                                                                                                                 │
│  Here is a comprehensive summary of the most impactful areas of AI application in healthcare:                   │
│                                                                                                                 │
│  ***                                                                                                            │
│                                                                                                                 │
│  ### 1. Enhanced Diagnostics and Imaging Analysis (The 'Seeing' Power)                                          │
│                                                                                                                 │
│  AI's most immediate and visible impact is in diagnostic capabilities. Deep learning algorithms, particularly   │
│  Convolutional Neural Networks (CNNs), are now capable of analyzing medical images—X-rays, MRIs, CT scans, and  │
│  pathology slides—with precision that often rivals, and in some limited tasks, surpasses, human experts.        │
│                                                                                                                 │
│  *   **Radiology:** AI systems are trained to detect subtle patterns indicative of early-stage diseases (e.g.,  │
│  identifying pulmonary nodules or signs of diabetic retinopathy) that might be missed by the human eye,         │
│  leading to earlier, less invasive diagnoses.                                                                   │
│  *   **Pathology:** AI accelerates the analysis of digitized tissue slides, quantifying cell abnormalities and  │
│  helping pathologists grade cancers, thereby standardizing and speeding up a complex and highly subjective      │
│  process.                                                                                                       │
│  *   **Insight:** AI serves as a "second pair of eyes" for clinicians, reducing diagnostic variability and      │
│  minimizing burnout related to the sheer volume of complex images reviewed daily.                               │
│                                                                                                                 │
│  ### 2. Accelerating Drug Discovery and Genomics (The 'Creating' Power)                                         │
│                                                                                                                 │
│  Perhaps the most revolutionary application is in the p

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Find interesting facts about AI in healthcare                                                            │
│  Agent: Senior Research Analyst                                                                                 │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Content Writer                                                                                          │
│                                                                                                                 │
│  Task: Write a short blog post about AI in healthcare                                                           │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Content Writer                                                                                          │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  ***                                                                                                            │
│                                                                                                                 │
│  ## The Doctor’s Co-Pilot: How AI Is Rewriting the Future of Healthcare                                         │
│                                                                                                                 │
│  When we talk about the future of medicine, it often feels like science fiction. But today, the deepest         │
│  breakthroughs aren't coming from a single breakthrough drug; they're coming from code.                         │
│                                                                                                                 │
│  Artificial Intelligence (AI) is not just a fancy new gadget for hospitals—it represents a fundamental shift    │
│  in how we approach health. We are moving away from a reactive model (waiting until we get sick to treat the    │
│  symptoms) and toward a powerful, proactive model: **predictive healthcare.**                                   │
│                                                                                                                 │
│  Think of AI as the ultimate co-pilot for every medical professional. It doesn't replace the doctor; it         │
│  enhances the doctor, arming them with superhuman powers of analysis, speed, and foresight.                     │
│                                                                                                                 │
│  Here is a look at the three ways AI is transforming care, from the diagnostic room to the research lab.        │
│                                                                                                                 │
│  ### 🔬 1. Seeing the Unseen: Diagnostic Superpowers                                                            │
│                                                                                                                 │
│  Perhaps AI’s most immediate impact is in reading images. Medical diagnosis—from X-rays to complex pathology    │
│  slides—requires massive expertise. AI systems are now trained on millions of these visuals, giving them a      │
│  capability that is nothing short of miraculous.                                                                │
│                                                                                                                 │
│  In radiology, AI acts as a "second pair of eyes," flagging incredibly subtle patterns—like faint nodules on a  │
│  lung scan or early signs of diabetic retinopathy—that might be easily missed by the human eye or buried in a   │
│  mountain of images. In pathology, AI analyzes tiny samples of tissue, standardizing the process of grading     │
│  diseases and speeding up results, making care more consistent and faster than ever before.                     │
│                                                                                                                 │
│  ### 🧬 2. Building the Cure: Accelerating Discovery                                                            │
│                                                                                                                 │
│  The traditional drug discovery process is notoriously sl

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Write a short blog post about AI in healthcare                                                           │
│  Agent: Content Writer                                                                                          │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Here's the result:
***

## The Doctor’s Co-Pilot: How AI Is Rewriting the Future of Healthcare

When we talk about the future of medicine, it often feels like science fiction. But today, the deepest breakthroughs aren't coming from a single breakthrough drug; they're coming from code.

Artificial Intelligence (AI) is not just a fancy new gadget for hospitals—it represents a fundamental shift in how we approach health. We are moving away from a reactive model (waiting until we get sick to treat the symptoms) and toward a powerful, proactive model: **predictive healthcare.**

Think of AI as the ultimate co-pilot for every medical professional. It doesn't replace the doctor; it enhances the doctor, arming them with superhuman powers of analysis, speed, and foresight.

Here is a look at the three ways AI is transforming care, from the diagnostic room to the research lab.

### 🔬 1. Seeing the Unseen: Diagnostic Superpowers

Perhaps AI’s most immediate impact is in reading images. Medical di

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: b7dab55e-93dc-4ec0-b241-5e9b4459c19c                                                                       │
│  Final Output: ***                                                                                              │
│                                                                                                                 │
│  ## The Doctor’s Co-Pilot: How AI Is Rewriting the Future of Healthcare                                         │
│                                                                                                                 │
│  When we talk about the future of medicine, it often feels like science fiction. But today, the deepest         │
│  breakthroughs aren't coming from a single breakthrough drug; they're coming from code.                         │
│                                                                                                                 │
│  Artificial Intelligence (AI) is not just a fancy new gadget for hospitals—it represents a fundamental shift    │
│  in how we approach health. We are moving away from a reactive model (waiting until we get sick to treat the    │
│  symptoms) and toward a powerful, proactive model: **predictive healthcare.**                                   │
│                                                                                                                 │
│  Think of AI as the ultimate co-pilot for every medical professional. It doesn't replace the doctor; it         │
│  enhances the doctor, arming them with superhuman powers of analysis, speed, and foresight.                     │
│                                                                                                                 │
│  Here is a look at the three ways AI is transforming care, from the diagnostic room to the research lab.        │
│                                                                                                                 │
│  ### 🔬 1. Seeing the Unseen: Diagnostic Superpowers                                                            │
│                                                                                                                 │
│  Perhaps AI’s most immediate impact is in reading images. Medical diagnosis—from X-rays to complex pathology    │
│  slides—requires massive expertise. AI systems are now trained on millions of these visuals, giving them a      │
│  capability that is nothing short of miraculous.                                                                │
│                                                                                                                 │
│  In radiology, AI acts as a "second pair of eyes," flagging incredibly subtle patterns—like faint nodules on a  │
│  lung scan or early signs of diabetic retinopathy—that might be easily missed by the human eye or buried in a   │
│  mountain of images. In pathology, AI analyzes tiny samples of tissue, standardizing the process of grading     │
│  diseases and speeding up results, making care more consistent and faster than ever before.                     │
│                                                                                                                 │
│  ### 🧬 2. Building the Cure: Accelerating Discovery                                                            │
│                                                                                                                 │
│  The traditional drug discovery process is notoriously s

In [ ]:
import os
from crewai import Agent, Task, Crew

## Create a simple txt file
work_dir = os.getenv("WORK_DIR")

# Importing crewAI tools
from crewai_tools import (
    DirectoryReadTool,
    FileReadTool,
    SerperDevTool,
    WebsiteSearchTool
)

# Instantiate tools
docs_tool = DirectoryReadTool(directory=f"{work_dir}/docs/text")
file_tool = FileReadTool()
search_tool = SerperDevTool()
web_rag_tool = WebsiteSearchTool()

# Create agents
researcher = Agent(
    role='Market Research Analyst',
    goal='Provide up-to-date market analysis of the AI industry',
    backstory='An expert analyst with a keen eye for market trends.',
    tools=[search_tool, web_rag_tool],
    verbose=True
)

writer = Agent(
    role='Content Writer',
    goal='Craft engaging blog posts about the AI industry',
    backstory='A skilled writer with a passion for technology.',
    tools=[docs_tool, file_tool],
    verbose=True
)

# Define tasks
research = Task(
    description='Research the latest trends in the AI industry and provide a summary.',
    expected_output='A summary of the top 3 trending developments in the AI industry with a unique perspective on their significance.',
    agent=researcher
)

write = Task(
    description="Write an engaging blog post about the AI industry, based on the research analyst's summary. Draw inspiration from the latest blog posts in the directory.",
    expected_output="A 4-paragraph blog post formatted in markdown with engaging, informative, and accessible content, avoiding complex jargon.",
    agent=writer,
    output_file=f'{work_dir}/outputs/L001/new_post.md'  # The final blog post will be saved here
)

# Assemble a crew with planning enabled
crew = Crew(
    agents=[researcher, writer],
    tasks=[research, write],
    verbose=True,
    planning=True,  # Enable planning feature
)

# Execute tasks
crew.kickoff()

In [8]:
from sentence_transformers import SentenceTransformer

# 1. Load a pretrained Sentence Transformer model
model = SentenceTransformer('sentence-transformers/all-mpnet-base-v2')
# The sentences to encode
sentences = [
    "The weather is lovely today.",
    "It's so sunny outside!",
    "He drove to the stadium.",
]

# 2. Calculate embeddings by calling model.encode()
embeddings = model.encode(sentences)
print(embeddings.shape)
# [3, 384]

# 3. Calculate the embedding similarities
similarities = model.similarity(embeddings, embeddings)
print(similarities)
# tensor([[1.0000, 0.6660, 0.1046],
#         [0.6660, 1.0000, 0.1411],
#         [0.1046, 0.1411, 1.0000]])

(3, 768)
tensor([[1.0000, 0.6817, 0.0492],
        [0.6817, 1.0000, 0.0421],
        [0.0492, 0.0421, 1.0000]])


In [6]:

from crewai import LLM
from pydantic import BaseModel

class Dog(BaseModel):
    name: str
    age: int
    breed: str

ollama_llm = LLM(
  model=f"ollama/{os.environ["OPENAI_MODEL_NAME"]}",
  base_url="http://localhost:11434/v1",
  temperature=0.8,
  response_format=Dog,
  api_key="ollama"  # Ollama doesn't require a real API key
)

response = ollama_llm.call(
    "Analyze the following messages and return the name, age, and breed. "
    "Meet Kona! She is 3 years old and is a black german shepherd."
)
print(response)

Name: Kona
Age: 3
Breed: German Shepherd


In [4]:
import asyncio
from crewai import LLM

async def stream_async():
    llm = LLM(
        model=f"ollama/{os.environ["OPENAI_MODEL_NAME"]}",
        base_url="http://localhost:11434/v1",
        temperature=0.8,
        stream=True,
        api_key="ollama"  # Ollama doesn't require a real API key
    )
    response = await llm.acall("Write a short story about AI")
    print(response)

#asyncio.run(stream_async())





In [5]:
await stream_async()

Output()

RecursionError: maximum recursion depth exceeded

In [ ]:
import nest_asyncio
nest_asyncio.apply()
asyncio.run(stream_async()) # Now will work


In [ ]:
#
loop = asyncio.get_running_loop()
loop.create_task(stream_async())

In [ ]:
# To execute a new event loop without interfering with the current one, you can run your async code in a background thread using ThreadPoolExecutor.
from concurrent.futures import ThreadPoolExecutor
with ThreadPoolExecutor() as executor:
    result = executor.submit(lambda: asyncio.run(stream_async())).result()

In [1]:
#
import os
import asyncio
from crewai import LLM

async def main():
    llm = LLM(
        model=f"ollama/{os.environ["OPENAI_MODEL_NAME"]}",
        base_url="http://localhost:11434/v1",
        temperature=0.8,
        response_format=Dog,
        stream=True,
        api_key="ollama"  # Ollama doesn't require a real API key
    )

    # Single async call
    response = await llm.acall("What is the capital of France?")
    print(response)

asyncio.run(main())

RuntimeError: asyncio.run() cannot be called from a running event loop

In [ ]:
from crewai.rag.config.utils import set_rag_config, get_rag_client, clear_rag_config
from chromadb.utils.embedding_functions.ollama_embedding_function import OllamaEmbeddingFunction
from crewai.rag.chromadb.types import ChromaEmbeddingFunctionWrapper
from typing import Literal, cast

# ChromaDB (default)
from crewai.rag.chromadb.config import ChromaDBConfig



set_rag_config(ChromaDBConfig(
    embedding_function=cast(
        ChromaEmbeddingFunctionWrapper,
        OllamaEmbeddingFunction(
            model_name="nomic-embed-text:latest",
            url="http://localhost:11434/api/embeddings"
        ),
    )
))

chromadb_client = get_rag_client()

# Qdrant
#from crewai.rag.qdrant.config import QdrantConfig
#set_rag_config(QdrantConfig())
#qdrant_client = get_rag_client()

# Example operations (same API for any provider)
client = chromadb_client  # or chromadb_client
client.create_collection(collection_name="docs")
client.add_documents(
    collection_name="docs",
    documents=[{"id": "1", "content": "CrewAI enables collaborative AI agents."}],
)
results = client.search(collection_name="docs", query="collaborative agents", limit=3)

clear_rag_config()  # optional reset

In [ ]:
from crewai_tools import PDFSearchTool

# - embedding_model (required): choose provider + provider-specific config
# - vectordb (required): choose vector DB and pass its config

tool = PDFSearchTool(
    pdf='/home/brijeshdhaker/IdeaProjects/bd-notebooks-module/docs/pdf/2026-04-01_BRIJESHD_PROFILE_MP.pdf',
    config={
        "embedding_model": {
            # Supported providers: "openai", "azure", "google-generativeai", "google-vertex",
            # "voyageai", "cohere", "huggingface", "jina", "sentence-transformer",
            # "text2vec", "ollama", "openclip", "instructor", "onnx", "roboflow", "watsonx", "custom"
            "provider": "openai",  # or: "google-generativeai", "cohere", "ollama", ...
            "config": {
                # Model identifier for the chosen provider. "model" will be auto-mapped to "model_name" internally.
                "model": "nomic-embed-text:latest",
                # Optional: API key. If omitted, the tool will use provider-specific env vars
                # (e.g., OPENAI_API_KEY or EMBEDDINGS_OPENAI_API_KEY for OpenAI).
                # "api_key": "sk-...",

                # Provider-specific examples:
                # --- Google Generative AI ---
                # (Set provider="google-generativeai" above)
                # "model_name": "gemini-embedding-001",
                # "task_type": "RETRIEVAL_DOCUMENT",
                # "title": "Embeddings",

                # --- Cohere ---
                # (Set provider="cohere" above)
                # "model": "embed-english-v3.0",

                # --- Ollama (local) ---
                # (Set provider="ollama" above)
                # "model": "nomic-embed-text",
            },
        },
        "vectordb": {
                    "provider": "chromadb",  # or "qdrant"
                    "config": {
                        # For ChromaDB: pass "settings" (chromadb.config.Settings) or rely on defaults.
                        # Example (uncomment and import):
                        # from chromadb.config import Settings
                        # "settings": Settings(
                        #     persist_directory="/content/chroma",
                        #     allow_reset=True,
                        #     is_persistent=True,
                        # ),

                        # For Qdrant: pass "vectors_config" (qdrant_client.models.VectorParams).
                        # Example (uncomment and import):
                        # from qdrant_client.models import VectorParams, Distance
                        # "vectors_config": VectorParams(size=384, distance=Distance.COSINE),

                        # Note: collection name is controlled by the tool (default: "rag_tool_collection"), not set here.
                    }
        },
    }
)


CREWAI_TOOLS_ALLOW_UNSAFE_PATHS is enabled — skipping file path validation for: /home/brijeshdhaker/IdeaProjects/bd-notebooks-module/docs/pdf/2026-04-01_BRIJESHD_PROFILE_MP.pdf


In [5]:
tool.run("What are the skills ?")

"Relevant Content:\n\nimproved release frequency from monthly to bi-weekly, ensuring on-time delivery against stringent regulatory \n\ndeadlines. \n\n \n\n• Led Data Engineering Chapter for IB Markets Regulatory Reporting, governing 5+ regulatory \n\nframeworks (MIFID-II, RTS23, EUBR/UKBR, SSD, SFTR) — overseeing ingestion, reconciliation, and distribution \n\nof deal-store data for ~10 million+ daily trade & transaction records with full audit trail and zero regulatory \n\nbreach incidents across a 3-year delivery window. \n\n \n\n\n\n\n\n\nPage 2:\n\n \n\n• Designed and executed migration strategy from Cloudera Data Platform to Azure Databricks, leveraging Azure \n\nData Factory, ADX, and Apache Flink — reducing data processing latency by ~50%, cutting infrastructure \n\nlicensing costs by ~35%, and enabling elastic scaling for peak regulatory reporting windows without manual \n\nintervention. \n\n \n\n• Partnered with business, operations, and risk teams to identify and remediate 3 

In [2]:
from openai import OpenAI

client = OpenAI(
    base_url='http://localhost:11434/v1/',
    api_key='ollama', # Required by SDK but ignored locally
)

In [3]:
response = client.embeddings.create(
    model="nomic-embed-text:latest",
    input="This is a test sentence to generate embeddings."
)

# Access the vector
embedding_vector = response.data[0].embedding
print(embedding_vector)

[0.009233055636286736, 0.04529495909810066, -0.16523373126983643, -0.050296466797590256, 0.07396187633275986, -0.03947746381163597, 0.04310481995344162, -0.01630023866891861, -0.018003441393375397, -0.07231365889310837, -0.018172480165958405, 0.0327409952878952, 0.06767822057008743, 0.05998419597744942, -0.03582586348056793, -0.009189702570438385, 0.03401781618595123, -0.07278216630220413, -0.05270470306277275, 0.061376843601465225, -0.02748437598347664, 0.01730821095407009, 0.001673025544732809, -0.03226184472441673, 0.0923922061920166, 0.007282547187060118, 0.02929798886179924, 0.012607570737600327, -0.0256004948168993, 0.007553429342806339, 0.01671944558620453, -0.008841334842145443, 0.005397442728281021, 0.0013583740219473839, -0.07328808307647705, -0.02308564819395542, 0.0010626226430758834, 0.04341905191540718, -0.006731040775775909, -0.034819476306438446, -0.003985951654613018, 0.0419052392244339, -0.031113505363464355, -0.014953446574509144, 0.004205641802400351, 0.002195752691